In [1]:
import os
import re
import pickle as pkl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import pandas as pd

In [2]:
from llm_unsupervised_conf.metrics import *
from sklearn.metrics import brier_score_loss, roc_auc_score

In [3]:
def check_baseline(model, dataset):

    print(f"{model} - {dataset}")

    oai_path = f"../outputs/_baselines_openai/{model}/{dataset}/baseline_{model}.csv"
    oai_df = pd.read_csv(oai_path).sort_values(by='id')
    # display(oai_df.head())

    scores = oai_df["confidence"]
    correct = oai_df["correct"]
    
    n_bins=12

    acc = np.mean(correct)
    ece1 = get_ece1(scores, correct, n_bins=n_bins)
    ece2 = get_ece2(scores, correct, n_bins=n_bins)
    mce = get_mce(scores, correct, n_bins=n_bins)
    brier = brier_score_loss(correct, scores)
    auroc = roc_auc_score(correct, scores)

    print("----"*10)
    print("-- * METRICS * --\n")
    print("acc:", acc)
    print("ECE1:", ece1)
    print("ECE2:", ece2)
    print("MCE:", mce)
    print("Brier:", brier)
    print("AUROC:", auroc, "\n")
    print("----"*30)

    results = {
        "model": model,
        "dataset": dataset,
        "Acc": acc,
        "ECE1": ece1,
        "ECE2": ece2,
        "MCE": mce,
        "Brier": brier,
        "AUROC": auroc,
    }
    return results

In [4]:
datasets = [
    "gsm8k",
    "polymath",
    "sciq",
    "trivia_qa",
    "webq",
]
models = [
    "gpt-4o-mini", 
]

all_results = []

# for model in models:
    # for dataset in datasets:

for dataset in datasets:
    for model in models:

        try:
        
            results = check_baseline(model, dataset)
            all_results.append(results)

        except:
            print(f"\n\nResults do not exist for {model} {dataset}\n\n")

df = pd.DataFrame(all_results)

gpt-4o-mini - gsm8k
----------------------------------------
-- * METRICS * --

acc: 0.303
ECE1: 0.6013000000000003
ECE2: 0.6054148190547074
MCE: 0.6229418221734213
Brier: 0.5701400000000001
AUROC: 0.5595243168506233 

------------------------------------------------------------------------------------------------------------------------
gpt-4o-mini - polymath
----------------------------------------
-- * METRICS * --

acc: 0.269
ECE1: 0.6313500000000003
ECE2: 0.6316629672275137
MCE: 0.72875
Brier: 0.5955525000000002
AUROC: 0.4994228001566322 

------------------------------------------------------------------------------------------------------------------------
gpt-4o-mini - sciq
----------------------------------------
-- * METRICS * --

acc: 0.686
ECE1: 0.23628999999999997
ECE2: 0.2749357761508038
MCE: 0.3721470019342289
Brier: 0.2607751000000001
AUROC: 0.7071943882193459 

-------------------------------------------------------------------------------------------------------------

In [5]:
df.sort_values(by="dataset")

,model,dataset,Acc,ECE1,ECE2,MCE,Brier,AUROC
0,gpt-4o-mini,gsm8k,0.303,0.60130,0.605415,0.622942,0.570140,0.559524
1,gpt-4o-mini,polymath,0.269,0.63135,0.631663,0.728750,0.595553,0.499423
2,gpt-4o-mini,sciq,0.686,0.23629,0.274936,0.372147,0.260775,0.707194
3,gpt-4o-mini,trivia_qa,0.804,0.12391,0.177466,0.278218,0.161793,0.761042
4,gpt-4o-mini,webq,0.553,0.35775,0.378137,0.452373,0.360633,0.683498


In [6]:
df.to_csv("../outputs/baseline_stats.csv", index=False)